# Session 3 - Break the Model

> **You have been handed a dataset and a model that reports a strong R-squared.**
> **Your brief: decide whether you would trust it, and back your answer with evidence.**

A retailer ran online ad campaigns across 120 region-months. For each, you have the monthly **ad spend** and the **new customers** acquired. Claude will happily fit a model and tell you it is good. Today your job is not to build a model. It is to **find out whether you can believe one.**

*Where the data comes from: a teaching dataset built for this course, so the right answer is known.*

**How to work**
- Solo or in pairs. Let Claude write the code; *you* decide whether the model holds up.
- Question every answer. A model you cannot break is not the same as a model that is right.

In [ ]:
# --- Load the data -------------------------------------------------
# The data loads from the course's public notebook repo, nothing to upload.
# No internet? Set DATA_URL = "" and upload the file when prompted (Colab only).
DATA_URL = "https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/main/session3/data/broken_model.csv"

import pandas as pd

if DATA_URL:
    df = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload broken_model.csv ...")
        up = files.upload()
        df = pd.read_csv(next(iter(up)))
    except ImportError:
        df = pd.read_csv("data/broken_model.csv")   # local fallback (repo: data/broken_model.csv)

df.head()

## What the columns mean

Each row is one region-month (120 in total).

| Column | Meaning |
|---|---|
| `ad_spend` | monthly online ad spend, in thousands of euro |
| `new_customers` | new customers acquired that month |

## Step 1 - Fit the model (let Claude drive)

A good prompt to Claude:

> *"Fit a simple linear regression of new_customers on ad_spend in Python using statsmodels, and print the full summary."*

Run the code it gives you:

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols('new_customers ~ ad_spend', data=df).fit()
print(model.summary())

Look at the R-squared and the p-value on `ad_spend`. Ask Claude whether this is a good model. **It will probably say yes** - a high R-squared and a tiny p-value. Note down what it claims, then keep going. We are about to test that claim.

## Step 2 - Predict before you compute  :dart:

Before you interrogate anything, commit to a view (write it in the next cell):

- Do you believe a **straight line** is the right shape for spend -> customers?
- In a real business, does each extra 1,000 euro of ad spend add the **same** number of customers, every time, for ever?
- If not, where would a straight-line model be **wrong**?

In [ ]:
# Your view before you look (just notes):
#
# Right shape for spend -> customers?  :
# Does each extra EUR1k add the same?   :
# Where would a straight line be wrong? :

## Step 3 - Interrogate the model

Two checks. Ask Claude for each, then read what you get back yourself.

**3a. Look at the relationship and the fitted line.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.scatter(df['ad_spend'], df['new_customers'], alpha=0.6, label='data')
xs = np.linspace(df['ad_spend'].min(), df['ad_spend'].max(), 100)
plt.plot(xs, model.params['Intercept'] + model.params['ad_spend'] * xs,
         color='orange', lw=2, label='straight-line model')
plt.xlabel('Ad spend (EUR 000s)'); plt.ylabel('New customers')
plt.legend(); plt.show()

**3b. Plot the residuals** (what the model missed) against its predictions. Healthy residuals are a shapeless cloud around zero.

In [ ]:
df['predicted'] = model.fittedvalues
df['residual'] = model.resid

plt.scatter(df['predicted'], df['residual'], alpha=0.6)
plt.axhline(0, color='red', ls='--')
plt.xlabel('Predicted new customers'); plt.ylabel('Residual'); plt.show()

**Read it.** Is the scatter shapeless, or can you see a **curve**? Does the spread stay even, or does it **fan out** as predictions get bigger? Either pattern means the straight-line model is missing something - whatever the R-squared said.

**3c. Train and test.** Build on 80% of the data, score on the 20% it never saw. A big gap means overfitting; similar scores mean the model generalises (which is *not* the same as being correct).

> Ask Claude: *"Split the data 80/20, fit the same regression on the training set, and report R-squared on both the training and the test set."*

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = df[['ad_spend']]; y = df['new_customers']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=1)
lin = LinearRegression().fit(Xtr, ytr)
print('Train R2:', round(r2_score(ytr, lin.predict(Xtr)), 3))
print('Test  R2:', round(r2_score(yte, lin.predict(Xte)), 3))

Here the train and test scores are **similar** - so the model is *not* overfitting. That is the trap: a model can pass the train/test check and still be wrong. The residual plot is what gives it away. **You need both tools, because they catch different problems.**

## Step 4 - Diagnose and fix

The residuals curve because the real relationship has **diminishing returns**: each extra euro of spend buys fewer customers than the last. A straight line cannot bend, so it is biased - it over-predicts in some places and under-predicts in others.

> Ask Claude: *"The relationship looks like it has diminishing returns. Refit the model using the square root of ad_spend as the predictor, and show me the new residual plot."*

In [ ]:
model2 = smf.ols('new_customers ~ np.sqrt(ad_spend)', data=df).fit()
print('Original R2:', round(model.rsquared, 3), '  Curved-fit R2:', round(model2.rsquared, 3))

plt.scatter(model2.fittedvalues, model2.resid, alpha=0.6)
plt.axhline(0, color='red', ls='--')
plt.xlabel('Predicted (curved model)'); plt.ylabel('Residual'); plt.show()

The curve in the residuals should be largely gone. (The fan-out may remain - that is the heteroscedasticity, a separate issue worth flagging. One fix rarely cures everything.)

**Optional - watch overfitting happen.** Ask Claude to fit a degree-10 polynomial of `ad_spend`, then redo the train/test check. The training R-squared will jump; watch what the test R-squared does.

## Step 5 - Your verdict (this is the point)

The straight-line model implied that **every extra 1,000 euro of spend adds the same number of customers** - so spend more, without limit. The data says otherwise: returns flatten, and past a point extra spend barely moves the needle. **Acting on the first model would waste budget.**

Write your verdict: would you trust the original model? In one or two sentences, what would you tell the marketing director - and what is the *decision* that changes once you know about diminishing returns?

In [ ]:
# Your verdict and recommendation:

## Before you leave

Jot these down for the debrief:
1. Which single check **exposed** the model - and what would you have missed without it?
2. The straight-line model's claim (constant returns) vs. what is really happening.
3. One sentence you would say to a client about trusting a model with a high R-squared.